In [1]:
import classifier
from scipy.optimize import curve_fit
import numpy as np

red, green, blue = '#ff0040', '#00aa00', '#187bff'

bistroke_times = {}
bistroke_freq = {}
bigrams = set()

with open(f"nstrokes/bistrokes_0.txt") as file:
    for l in file:
        layout, bigram, freq, *times = l.split("\t")

        if bigram not in bigrams:
            bigrams.add(bigram)
            
        bistroke_freq[(layout, bigram)] = int(freq)
        bistroke_times[(layout, bigram)] = [
            list(map(int, t.strip()[1:-1].split(", "))) for t in times
        ]

In [2]:
layout = "qwerty"

freqs = []
times = []
is_sfb = []
home_row = []
not_homerow = []
delta_x = []
is_bottom = []
top1 = []
home1 = []
bottom1 = []
top2 = []
home2 = []
bottom2 = []
finger1 = []
finger2 = []
same_hand = []
is_pinky1 = []
is_pinky2 = []
is_ring1 = []
is_ring2 = []
is_middle1 = []
is_middle2 = []
is_index1 = []
is_index2 = []
dists = []
y_dist = []
c = []

def get_iqr_avg(data):
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    IQR = Q3-Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    
    new_data = [x for x in data if x >= lower_bound and x <= upper_bound]
    print(len(data), len(new_data))
    
    return sum(new_data)/len(new_data)


for bg in bigrams:
    if not any([c in '!@#$%^&*()QWERTYUIOP{}|ASDFGHJKL:"ZXCVBNM<>?' for c in bg]):
        if (layout, bg) in bistroke_freq:
            time_data = [t[1] for t in bistroke_times[(layout, bg)] if t[0] > 0 and t[1] < 1000]
            x_val = bistroke_freq[(layout, bg)]
            y_val = get_iqr_avg(time_data)
            
            if x_val != 0:
                freqs.append(x_val)
                times.append(y_val)
                is_sfb.append(int(classifier.same_finger(bg)))
                delta_x.append(classifier.get_dx(bg))
                top1.append(classifier.kb.get_row(bg[0]) == 3)
                home1.append(classifier.kb.get_row(bg[0]) == 2)
                bottom1.append(classifier.kb.get_row(bg[0]) == 1)
                top2.append(classifier.kb.get_row(bg[1]) == 3)
                home2.append(classifier.kb.get_row(bg[1]) == 2)
                bottom2.append(classifier.kb.get_row(bg[1]) == 1)
                y_dist.append(classifier.get_dy(bg))
                same_hand.append(classifier.same_hand(bg))

                # Finger stuff
                finger1.append(abs(classifier.kb.get_col(bg[0])))
                finger2.append(abs(classifier.kb.get_col(bg[1])))
                
                is_pinky1.append(abs(classifier.kb.get_col(bg[0])) == 5)
                is_pinky2.append(abs(classifier.kb.get_col(bg[1])) == 5)
                is_ring1.append(abs(classifier.kb.get_col(bg[0])) == 4)
                is_ring2.append(abs(classifier.kb.get_col(bg[1])) == 4)
                is_middle1.append(abs(classifier.kb.get_col(bg[0])) == 3)
                is_middle2.append(abs(classifier.kb.get_col(bg[1])) == 3)
                is_index1.append(abs(classifier.kb.get_col(bg[0])) in (1,2))
                is_index2.append(abs(classifier.kb.get_col(bg[1])) in (1,2))

                c.append(red if is_sfb[-1] else blue if same_hand[-1] else green)

17282 15769
14232 13240
59861 54528
48796 46445
24782 22857
39969 36340
55852 50723
40969 38133
119554 109488
656608 612388
29725 27276
200000 190566
1319 1214
595 557
3894 3643
74470 68892
9795 9120
91155 83369
13382 12929
558 530
76822 71397
554 533
557 520
2879 2707
9086 8369
1315 1235
1745 1623
84249 76376
606 548
13816 12727
30501 28642
3150 2900
4382 4085
1162 1137
1180 1109
71524 65856
198261 188040
54657 50222
448523 420398
125477 116565
37205 34788
5216 4808
17139 15372
5710 5259
116662 107095
17673 15833
71686 66424
3890 3546
552 511
8732 8118
3210 2945
1213 1169
74677 70047
3680 3481
67436 61831
9476 8595
51859 47056
3841 3589
8004 7279
49861 45568
66255 62502
22263 20274
23213 21654
640 615
60219 54661
203952 190184
45302 42027
89753 82020
264677 253303
15703 14295
8951 8270
93273 85050
15278 14120
39869 36435
88581 80021
15546 14148
1693 1605
20621 18807
20388 18900
429 426
236247 211813
63000 57181
2056 2004
675 621
440284 403177
41418 37326
1250 1174
57293 51748
1276 122

In [122]:
# def model_function(x, a, b, c, d, e, f, g, h, i, j, k, l):
#     freq_pen = (a * np.log(x[0] + b) + c)
#     finger_pen = (d*x[1]+e*x[2]+f*x[3]+g*x[4] + h)
#     
#     return freq_pen*finger_pen
#     return (a * np.log(x[0] + b) + c) * (d * x[1] + e * x[2] + g * x[3] + f)

# freqs, is_pinky1, is_ring1, is_middle1, is_index1, is_pinky2, is_ring2, is_middle2, is_index2, is_sfb, delta_x, y_dist, same_hand, bottom2, home2, top2
def model_function(features, a, b, c, d, e, f, g, h, i, j, k, l, m, n, o, p, q, r, t, u, v, w, x, y, z, A, B, C, D, E, F, G, H, I, J, K, L, M, N, O, P, Q, R, S, T, U, V, W, X, Y, Z, a2, b2, c2, d2, e2, f2, g2): # 
    freq, is_sfb, dx, dy, same_hand = features[0], features[9], features[10], features[11], features[12]
    bottom1, home1, top1, bottom2, home2, top2 = features[13:19]
    freq_pen = (a * np.log(freq + b) + c)
    
    # sfb_finger_pen = (d*features[5]+e*features[6]+f*features[7]+g*features[8] + h)
    row_pen1 = (d*bottom2+e*home2+f*top2+g)
    row_pen2 = (h*bottom2+A*home2+B*top2+C)
    row_pen2b = (X*bottom1+Y*home1+Z*top1+a2)
    row_pen3 = (I*bottom2+J*home2+K*top2+L)
    row_pen3b = (b2*bottom1+c2*home1+d2*top1+e2)

    col_pen1 = (t*features[5]+u*features[6]+v*features[7]+w*features[8] + x)
    col_pen2 = (k*features[5]+l*features[6]+m*features[7]+n*features[8] + o)
    col_pen2b = (N*features[1]+O*features[2]+P*features[3]+Q*features[4] + R)
    col_pen3 = (D*features[5]+E*features[6]+F*features[7]+G*features[8] + H)
    col_pen3b = (S*features[1]+T*features[2]+U*features[3]+V*features[4] + W)

    finger_pen1 = col_pen1*row_pen1
    finger_pen2 = col_pen2*col_pen2b*row_pen2*row_pen2b
    finger_pen3 = col_pen3*col_pen3b*row_pen3*row_pen3b

    dist = (dx**2+dy**2)**0.5
    sfb_pen = (i*is_sfb*(dist+finger_pen1)+j)
    scissor_pen = (p*(1-is_sfb)*same_hand*((L+dy*M)+finger_pen2)+q) # 
    alt_boost = y*(1-same_hand)*finger_pen3+z

    return freq_pen*(alt_boost+sfb_pen+scissor_pen)




"""sfb mode
def model_function(features, a, b, c, d, e, f, g, h, i, j, k, l, m, n, o, p, q, r, t, u, v, w, x, y, z, aa, ab, ac, ad, ae, af, ag, ah, aj, ai, ak, al, am, an): # 
    freq, is_sfb, dx, dy, same_hand = features[0], features[9], features[10], features[11], features[12]
    bottom, home, top = features[13:16]
    freq_pen = (a * np.log(freq + b) + c)
    
    # sfb_finger_pen = (d*features[5]+e*features[6]+f*features[7]+g*features[8] + h)
    row_pen1 = (d*bottom+e*home+f*top+g)
    row_pen2 = (h*bottom+aa*home+ab*top+ac)
    row_pen3 = (ai*bottom+aj*home+ak*top+al)

    finger_pen1 = (t*features[5]+u*features[6]+v*features[7]+w*features[8] + x)
    finger_pen2 = (k*features[5]+l*features[6]+m*features[7]+n*features[8] + o)
    finger_pen3 = (ad*features[5]+ae*features[6]+af*features[7]+ag*features[8] + ah)
    #finger_pen = finger_pen1+finger_pen2

    dist = (dx**2+dy**2)**0.5
    sfb_pen = (i*is_sfb*dist*(finger_pen1+row_pen1)+j)
    scissor_pen = ((1-is_sfb)*p*(1/(dx+r))*(finger_pen2+row_pen2)*same_hand*(al+dy*am)+q)
    alt_boost = y*(1-same_hand)*(finger_pen3+row_pen3)+z

    return freq_pen+alt_boost+sfb_pen+scissor_pen
"""

freqs, times, is_sfb, delta_x, is_pinky1, is_ring1, is_middle1, is_index1, is_pinky2, is_ring2, is_middle2, is_index2, y_dist, same_hand, bottom1, home1, top1, bottom2, home2, top2, c = zip(*sorted(zip(freqs, times, is_sfb, delta_x, is_pinky1, is_ring1, is_middle1, is_index1, is_pinky2, is_ring2, is_middle2, is_index2, y_dist, same_hand, bottom1, home1, top1, bottom2, home2, top2, c), key = lambda x: x[0]))

freqs = np.array(freqs, dtype=np.float64)
times = np.array(times, dtype=np.float64)
is_sfb = np.array(is_sfb, dtype=np.float64)
delta_x = np.array(delta_x, dtype=np.float64)
top1 = np.array(top1, dtype=np.float64)
home1 = np.array(home1, dtype=np.float64)
bottom1 = np.array(bottom1, dtype=np.float64)
top2 = np.array(top2, dtype=np.float64)
home2 = np.array(home2, dtype=np.float64)
bottom2 = np.array(bottom2, dtype=np.float64)
finger1 = np.array(finger1, dtype=np.float64)
finger2 = np.array(finger2, dtype=np.float64)
is_pinky1 = np.array(is_pinky1, dtype=np.float64)
is_pinky2 = np.array(is_pinky2, dtype=np.float64)
is_ring1 = np.array(is_ring1, dtype=np.float64)
is_ring2 = np.array(is_ring2, dtype=np.float64)
is_middle1 = np.array(is_middle1, dtype=np.float64)
is_middle2 = np.array(is_middle2, dtype=np.float64)
is_index1 = np.array(is_index1, dtype=np.float64)
is_index2 = np.array(is_index2, dtype=np.float64)
dists = np.array(dists, dtype=np.float64)
y_dist= np.array(y_dist, dtype=np.float64)
same_hand= np.array(same_hand, dtype=np.float64)

input_data = [freqs, is_pinky1, is_ring1, is_middle1, is_index1, is_pinky2, is_ring2, is_middle2, is_index2, is_sfb, delta_x, y_dist, same_hand, bottom1, home1, top1, bottom2, home2, top2]
popt, pcov = curve_fit(model_function, input_data, times, maxfev=50000)

/tmp/ipykernel_37566/2017009630.py:12: RuntimeWarning: invalid value encountered in log
  freq_pen = (a * np.log(freq + b) + c)
/usr/local/lib/python3.8/dist-packages/scipy/optimize/_minpack_py.py:906: OptimizeWarning: Covariance of the parameters could not be estimated
  warnings.warn('Covariance of the parameters could not be estimated',


In [123]:
%matplotlib qt

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap


mycmap = LinearSegmentedColormap.from_list('custom_colormap', list(reversed(['#fff829', '#f4f730', '#e8f737', '#ddf63f', '#d1f546', '#c5f34d', '#b9f254', '#acf15b', '#a0ef62', '#92ed68', '#85ec6e', '#76ea74', '#66e77a', '#55e580', '#41e385', '#24e08b', '#00dd90', '#00da95', '#00d799', '#00d49e', '#00d1a2', '#00cda6', '#00caaa', '#00c6ae', '#00c2b1', '#00beb4', '#00bab7', '#00b6ba', '#00b2bc', '#00adbf', '#00a9c0', '#00a4c2', '#00a0c3', '#009bc4', '#0096c5', '#0091c6', '#008dc6', '#0088c6', '#0083c5', '#007ec4', '#0079c3', '#0074c2', '#006fc0', '#0069be', '#0064bc', '#005fb9', '#145ab6', '#2155b3', '#2950b0', '#304bac', '#3547a8', '#3942a4', '#3d3d9f', '#40389b', '#423396', '#442f91', '#462a8b', '#472586', '#482080', '#491c7a', '#491775', '#49116f', '#490c68', '#490562'])))

jump = 1

plt.figure()
new_y = model_function([arr[::jump] for arr in input_data], *popt)
scatter = plt.scatter(freqs[::jump], times[::jump], c=c[::jump], s=50)

plt.plot(freqs[::jump], new_y, c="black")
#plt.axhline(0, color='black', linewidth=0.5)
#plt.scatter(freqs, times-new_y, c="red")
plt.xlabel("Number of Occurrences in Corpus ")
plt.ylabel("Average Typing Time (Milliseconds)")
plt.xscale("log")

plt.show()

sum_of_squares = np.sum((times - np.mean(times))**2)

residuals = times-new_y
print("R^2:", 1 - np.sum((residuals)**2)/sum_of_squares)
print("MAE:", np.mean(np.abs(residuals)))

R^2: 0.7555474255717182
MAE: 25.376422151429516
